In [1]:
import os
import sys

# VS Code'un eksik aldığı yolları manuel olarak Windows'a tanıtıyoruz
torch_lib_path = r"C:\Users\Mustafa\s_env\Lib\site-packages\torch\lib"
env_scripts_path = r"C:\Users\Mustafa\s_env\Scripts"

os.environ['PATH'] = torch_lib_path + ";" + env_scripts_path + ";" + os.environ.get('PATH', '')
os.add_dll_directory(torch_lib_path)

import torch
print("PyTorch Başarıyla Yüklendi! Versiyon:", torch.__version__)

PyTorch Başarıyla Yüklendi! Versiyon: 2.13.0+cu126


In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModel, Trainer, TrainingArguments, EarlyStoppingCallback, TrainerCallback
)
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
import mlflow
import re
import os

mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("transformer_finetuning")

TASK_COLS = ["type", "queue", "category", "priority"]
MODEL_NAME = "xlm-roberta-base"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

c:\Users\Mustafa\s_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Mustafa\s_env\Lib\site-packages\pydantic\_internal\_fields.py:132: UserWarning: Field "model_name" in PromptModelConfig has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


Device: cuda


In [3]:
def light_clean(t):
    if not isinstance(t, str):
        return ""
    t = t.replace("\\n", " ")
    t = re.sub(r"<[^>]+>", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

train_df = pd.read_json("../data/processed/train.jsonl", lines=True)
val_df = pd.read_json("../data/processed/val.jsonl", lines=True)
train_df["body_light_clean"] = train_df["body"].apply(light_clean)
val_df["body_light_clean"] = val_df["body"].apply(light_clean)

In [4]:
encoders = {}
y_train_dict, y_val_dict, class_weights = {}, {}, {}

for col in TASK_COLS:
    le = LabelEncoder()
    y_train_dict[col] = le.fit_transform(train_df[col])
    y_val_dict[col] = le.transform(val_df[col])
    encoders[col] = le
    classes = np.arange(len(le.classes_))
    w = compute_class_weight(class_weight="balanced", classes=classes, y=y_train_dict[col])
    class_weights[col] = torch.tensor(w, dtype=torch.float32).to(device)

num_classes_dict = {col: len(encoders[col].classes_) for col in TASK_COLS}

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LENGTH = 128

class TicketDataset(Dataset):
    def __init__(self, texts, y_dict, tokenizer, max_length):
        self.texts = texts
        self.y_dict = y_dict
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], truncation=True, padding="max_length",
            max_length=self.max_length, return_tensors="pt"
        )
        item = {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
        }
        for col in TASK_COLS:
            item[col] = torch.tensor(self.y_dict[col][idx], dtype=torch.long)
        return item

train_ds = TicketDataset(train_df["body_light_clean"].values, y_train_dict, tokenizer, MAX_LENGTH)
val_ds = TicketDataset(val_df["body_light_clean"].values, y_val_dict, tokenizer, MAX_LENGTH)

c:\Users\Mustafa\s_env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Mustafa\.cache\huggingface\hub\models--xlm-roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [6]:
class MultiTaskTransformer(nn.Module):
    def __init__(self, model_name, num_classes_dict, class_weights=None):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.task_names = list(num_classes_dict.keys())
        self.heads = nn.ModuleDict({f"head_{t}": nn.Linear(hidden, n) for t, n in num_classes_dict.items()})
        self.class_weights = class_weights or {}

    def forward(self, input_ids=None, attention_mask=None, **task_labels):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(outputs.last_hidden_state[:, 0])
        logits = {t: self.heads[f"head_{t}"](pooled) for t in self.task_names}
        loss = None
        provided = {t: task_labels[t] for t in self.task_names if t in task_labels and task_labels[t] is not None}
        if provided:
            loss = sum(
                nn.functional.cross_entropy(logits[t], labels, weight=self.class_weights.get(t))
                for t, labels in provided.items()
            )
        logits_tuple = tuple(logits[t] for t in self.task_names)
        return {"loss": loss, "logits": logits_tuple}


class MultiTaskTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(**inputs)
        return (outputs["loss"], outputs) if return_outputs else outputs["loss"]


def compute_metrics(eval_pred):
    logits_tuple, label_ids = eval_pred
    result = {}
    for i, t in enumerate(TASK_COLS):
        preds = np.argmax(logits_tuple[i], axis=1)
        result[f"{t}_accuracy"] = (preds == label_ids[i]).mean()
    return result


class MLflowCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            for k, v in logs.items():
                if isinstance(v, (int, float)):
                    mlflow.log_metric(k, v, step=state.global_step)

In [ ]:
os.makedirs("../models", exist_ok=True)

model = MultiTaskTransformer(MODEL_NAME, num_classes_dict, class_weights)

training_args = TrainingArguments(
    output_dir="../models/xlmr_multitask_checkpoints",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    num_train_epochs=5,
    learning_rate=2e-5,
    lr_scheduler_type="linear",
    warmup_steps=100,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    remove_unused_columns=False,
    label_names=TASK_COLS,
    report_to=[],
)

trainer = MultiTaskTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2), MLflowCallback()],
)

with mlflow.start_run(run_name="xlmr_multitask"):
    mlflow.log_param("model_name", MODEL_NAME)
    mlflow.log_param("max_length", MAX_LENGTH)
    mlflow.log_param("batch_size", 16)
    trainer.train()

    metrics = trainer.evaluate()
    print(metrics)

    mlflow.pytorch.log_model(model, name="model", serialization_format="pickle")

  1%|▏         | 50/3580 [01:45<2:08:59,  2.19s/it]

{'loss': 6.9032, 'grad_norm': 39.25565719604492, 'learning_rate': 9.200000000000002e-06, 'epoch': 0.07}


  3%|▎         | 100/3580 [03:29<1:56:59,  2.02s/it]

{'loss': 6.2625, 'grad_norm': 54.908790588378906, 'learning_rate': 1.9200000000000003e-05, 'epoch': 0.14}


  4%|▍         | 150/3580 [05:12<1:56:59,  2.05s/it]

...

 92%|█████████▏| 3300/3580 [1:55:09<09:22,  2.01s/it]

{'loss': 2.9456, 'grad_norm': 101.38892364501953, 'learning_rate': 1.6436781609195405e-06, 'epoch': 4.61}


 94%|█████████▎| 3350/3580 [1:56:51<07:41,  2.01s/it]

{'loss': 2.7892, 'grad_norm': 90.30406951904297, 'learning_rate': 1.35632183908046e-06, 'epoch': 4.68}


 95%|█████████▍| 3387/3580 [1:58:12<07:07,  2.22s/it]

In [ ]:
import mlflow
import torch

# 1. Az önce biten eğitimin Run ID'sini al
run_id = mlflow.last_active_run().info.run_id

# 2. O eğitimin içine tekrar bağlan
with mlflow.start_run(run_id=run_id):
    # Trainer'dan modeli temizle 
    unwrapped_model = trainer.accelerator.unwrap_model(model)

    # Ağırlıkları geçici dosyaya kaydetme
    torch.save(unwrapped_model.state_dict(), "temp_clean_weights.pt")

    # Yeni, temiz bir model objesi yaratma
    clean_model = MultiTaskTransformer(MODEL_NAME, num_classes_dict, class_weights)

    # Eğitilen ağırlıkları buna yükleme
    clean_model.load_state_dict(torch.load("temp_clean_weights.pt"))

    # Temiz modeli MLflow'a kaydetme
    mlflow.pytorch.log_model(clean_model, "model", serialization_format="pickle")

# 3. Model Registry'ye (Staging) aktarma
model_uri = f"runs:/{run_id}/model"
result = mlflow.register_model(model_uri, "customer_ticket_xlmr_multitask")

client = mlflow.MlflowClient()
client.set_registered_model_alias("customer_ticket_xlmr_multitask", "staging", result.version)
print(f"XLM-R Model kaydedildi: version {result.version}, alias='staging'")

In [ ]:
run_id = mlflow.last_active_run().info.run_id
model_uri = f"runs:/{run_id}/model"

result = mlflow.register_model(model_uri, "customer_ticket_xlmr_multitask")

client = mlflow.MlflowClient()
client.set_registered_model_alias("customer_ticket_xlmr_multitask", "staging", result.version)
print(f"Model kaydedildi: version {result.version}, alias='staging'")